In [21]:
import duckdb
import dimcli
import pandas as pd


In [ ]:
# Connect to the database
con = duckdb.connect("publications.db")

In [ ]:
# Have a look at the data stored in the csv file
con.sql("SELECT PublicationID, Title, Abstract, Year, Pillar, Research_category  FROM 'publications_test_data.csv'").show()

┌────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# This would be a way to create the table and load the data in one step, but it doesn't allow us to set the PublicationID as the primary key
# con.sql("CREATE TABLE publications_raw AS SELECT PublicationID, Title, Abstract, Pillar, Research_category FROM 'publications_data.csv'")

# This line deletes the created table
# con.sql("DROP TABLE IF EXISTS publications_raw")

In [ ]:
# In order to have the Publication ID as the primary key, we need to create the table in two steps:
# Step 1: create the table with the schema
con.sql("""
    CREATE TABLE publications_raw (
        id VARCHAR PRIMARY KEY,
        title VARCHAR,
        abstract VARCHAR,
        year INT,
        pillar VARCHAR,
        research_category VARCHAR
    )
""")

# Step 2: load the CSV into it
con.sql("INSERT INTO publications_raw SELECT PublicationID, Title, Abstract, Year, Pillar, Research_category FROM 'publications_test_data.csv'")

In [31]:
con.sql("DESCRIBE publications_raw").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ title             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ pillar            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ research_category │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [ ]:
# Dimensions API login using dsl.ini file
dimcli.login()

Searching config file credentials for default 'live' instance..


Dimcli - Dimensions API Client (v1.7)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.15
Method: dsl.ini file


In [ ]:
# Make query
%%dsldf
search publications
for """("meat substitute" OR "meat analogue")"""
where year in [2020:2025]
return publications[id+title+abstract+year]
limit 100

Returned Publications: 100 (total = 16616)
Time: 1.86s


,id,title,abstract,year
0,pub.1187231244,Plant-proteins based 3D meat analog printing: ...,This review summarizes and critically analyze ...,2025
1,pub.1182395108,Market Status of Meat Analogs and Their Impact...,"The alternative meat industry, which aims to r...",2024
2,pub.1182772853,Plant Taxa as Raw Material in Plant-Based Meat...,<b>Background/Objectives</b>: The environmenta...,2024
3,pub.1192791591,Mushroom: an emerging source for next generati...,"Background: In recent years, plant-based and a...",2025
4,pub.1176097424,Plant-based Meat Analogs: Perspectives on Thei...,Purpose of ReviewPlant-based meat analogs (PBM...,2024
...,...,...,...,...
95,pub.1176270921,Spirulina platensis protein-based emulsion gel...,"In this paper, Spirulina platensis protein-bas...",2024
96,pub.1190536409,A Dimensionless Empirical Model to Predict Hea...,An empirical dimensionless relationship useful...,2025
97,pub.1169992045,Construction of a soybean protein isolate/poly...,A growing interest has arisen in recreating re...,2024
98,pub.1195785564,Aggregation mechanism of soybean protein isola...,The aggregation behavior and interaction mecha...,2025


In [ ]:
# Create new table within database for query results
con.sql("""
    CREATE TABLE publications_query (
        id VARCHAR PRIMARY KEY, 
        title VARCHAR, 
        abstract VARCHAR, 
        year INT)
    """)

In [ ]:
# Insert query results into database
con.sql("INSERT INTO publications_query SELECT * FROM dsl_last_results")

In [63]:
# Extract query results that had not been present in the database yet
con.sql("""
    SELECT COUNT(*)
    FROM publications_raw
    WHERE id IN (SELECT id FROM publications_query)
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           23 │
└──────────────┘



In [58]:
# Add a column for in/out of scope
con.sql("ALTER TABLE publications_raw ADD COLUMN scope TEXT DEFAULT 'out'")

# Update a value
con.sql("UPDATE publications_raw SET scope = 'in' WHERE id IN (SELECT id FROM publications_query)")

In [59]:
# Convert to pandas dataframe
df = con.sql("SELECT * FROM publications_raw").df()

In [ ]:
# Filter for in scope publications
df[df['scope'] == 'in']

,id,title,abstract,year,pillar,research_category,scope
1955,pub.1167674511,Structural and mechanical anisotropy in plant-...,The rising demand for plant-based meat analogu...,2024,PB,Manufacturing (incl Texturization methods),in
2087,pub.1169358163,Influence of Lowering the pH Value on the Gene...,High-moisture extrusion of plant proteins to c...,2024,PB,Manufacturing (incl Texturization methods),in
2137,pub.1170440622,In vitro digestibility and solubility of phosp...,Interest in plant-based meat analogues has inc...,2024,PB,End product formulation,in
2171,pub.1171008316,Exploring the Role and Functionality of Ingred...,The development of plant-based meat analogues ...,2024,PB,End product formulation,in
2305,pub.1173139999,Protein transition: focus on protein quality i...,"The current consumption trends, combined with ...",2024,CC,Food safety & quality,in
2381,pub.1174056169,Plant-Based Meat Analogues: Exploring Proteins...,As the lack of resources required to meet the ...,2024,PB,End product formulation,in
2482,pub.1175387236,Meat substitutes in Swedish school meals: nutr...,This study provides an overview of the ingredi...,2024,PB,Health /nutrition,in
2547,pub.1181205352,Alternative proteins; A path to sustainable di...,With a growing global population and the resul...,2024,CC,End product formulation,in
2578,pub.1181632383,Comparing the nutritional value and prices of ...,Since overconsumption of animal-sourced foods ...,2024,CC,Consumer and market research,in
2663,pub.1182670243,Is it still meat? The effects of replacing mea...,Consumer interest in a shift toward moderating...,2024,CC,No sector assigned,in


In [64]:
con.close()